## 🍽️ Restaurant Inventory Optimization — Demand Forecasting & Waste Reduction

### Problem Statement

Quick service restaurants operate on thin margins where inventory mismanagement directly eats into profitability. 
Over-ordering perishable ingredients leads to significant waste costs, while under-ordering causes stockouts, lost sales,
and expensive emergency purchases.
This project addresses a real operational challenge faced by quick service restaurant chains:
how do you know exactly how much of each ingredient to order every week?

### The Challenge

Given historical ordering and sales data for a quick service restaurant, build a system that:
1. Predicts weekly ingredient demand for the next "N" weeks per ingredient
2. Recommends optimal order quantities that minimize waste without risking stockouts
3. Quantifies the financial impact how much money does smarter ordering actually save?
4. Identifies the best supplier for each ingredient based on capacity and cost
   
These datasets have been provided: historical_order_summary,historical_sales,key_cost_items,recipe,supplier_details

### Importing Required libraries

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


In [ ]:
# reading the historical_order dataset
historical_order=pd.read_csv("historical_order_summary.csv")


In [ ]:
# Getting a view of the dataset
historical_order.info()
historical_order.head()

##### From the above table, each column is missing 11 rows of data. That will be handled by dropping those rows.
##### All columns are also of the datatype object. The data type of some columns will be changed 
##### The "Quantity" column also has a column between the values which we will have to eliminate

In [ ]:
# This drops all rows which are empty
historical_order.dropna(inplace=True)
# This returns rows where the column does NOT contain numbers
historical_order[~historical_order["Order made On"].str.contains(r"\d", na=False)]
# Removes rows where the value is literally "Order made On"
historical_order = historical_order[historical_order["Order made On"] != "Order made On"]
# This line helps to remove commas and spaces
historical_order["Quantity"] = historical_order["Quantity"].str.replace(",", "").str.strip()
# This helps to remove commas, dollar sign, removes extra spaces
historical_order["Cost/unit"] = historical_order["Cost/unit"].str.replace(",", "").str.replace("$", "").str.strip()

In [ ]:
# Converting the data types to numeric and date time formats
historical_order["Quantity"] = pd.to_numeric(historical_order["Quantity"], errors="coerce")
historical_order["Lead Time"] = pd.to_numeric(historical_order["Lead Time"], errors="coerce")
historical_order["Cost/unit"] = pd.to_numeric(historical_order["Cost/unit"], errors="coerce")
historical_order["Order made On"] = pd.to_datetime(historical_order["Order made On"], format="mixed", errors="coerce")
historical_order["Order received On"] = pd.to_datetime(historical_order["Order received On"], format="mixed", errors="coerce")

In [ ]:
# renaming columns
historical_order = historical_order.rename(columns={
    "Order made On": "order_date",
    "Order received On": "Delivery_date",
})

In [ ]:
historical_order.head(5)

#### We will now work on the Historical sales dataset

In [ ]:
# reading the dataset
historical_sales=pd.read_csv("historical_sales.csv")
historical_sales.head()

#### Similar to the previous dataset there are also comma's present in the Unit column

In [ ]:
# converting the week column to datetime and Units sold to numeric values
historical_sales["Week Starting"] = pd.to_datetime(historical_sales["Week Starting"], dayfirst=True,format="mixed")
# This line helps to remove commas and spaces
historical_sales["Units sold"] = historical_sales["Units sold"].str.replace(",", "").str.strip()
# This helps to remove commas, dollar sign, removes extra spaces
historical_sales["Units sold"] = historical_sales["Units sold"].str.replace(",", "").str.replace("$", "").str.strip()
# Converts the units column to integer datatype
historical_sales["Units sold"]=pd.to_numeric(historical_sales["Units sold"])

#### Cleaning the "item_cost" Dataset

In [ ]:
# reading the dataset
item_cost=pd.read_csv("key_cost_items.csv")
item_cost.head(12)

#### From the above there are empty colunms and rows which will be dropped and there are also different cateories of cost(Holding,Waste) which we will classify with a new column

In [ ]:
# first we drop the two empty columns
item_cost.drop(columns=["Unnamed: 2","Unnamed: 3"],inplace=True)
# we rename the columns as well
item_cost=item_cost.rename(columns={
    "Key Item":"Item",
    "Cost/Unit per day":"cost_per_unit"
})
# changing data types
item_cost["cost_per_unit"] = pd.to_numeric(item_cost["cost_per_unit"], errors="coerce")
item_cost["Item"] = item_cost["Item"].astype(str)

In [ ]:
# extracting the ingredients
item_cost["ingredient"] = item_cost["Item"].str.extract(r"- (.*)")

In [ ]:
# drops empty columns we don't need
item_cost.dropna(inplace=True)

In [ ]:
# Computing the cost type
item_cost["cost_type"]=item_cost["Item"].apply(
    lambda x :"holding" if "Holding" in x else
                "waste" if "Waste" in x else
                "selling_price"
)

In [ ]:
item_cost.drop(columns=["Item"],inplace=True)

In [ ]:
item_cost

#### Cleaning the "data_recipe" Dataset

In [ ]:
# reading the data
data_recipe=pd.read_csv("recipe.csv")

#### For consistency through-out our work we will be replacing "Green Chillies" with "Chillies"

In [ ]:
# replacing the term green chillies
data_recipe=data_recipe.replace("Green Chillies","Chillies")

In [ ]:
# coverting data types
data_recipe["Ingredient"] = data_recipe["Ingredient"].astype("string")
# renaming columns
data_recipe=data_recipe.rename(columns={
    "Quantity (in respective unit of measure)":"Quantity",
    "Shelf_Life (in days)":"Shelf life"
})

In [ ]:
data_recipe.head()

#### Cleaning the "data_supplier" Dataset

In [ ]:
# reading the data
data_supplier=pd.read_csv("supplier_details.csv")
data_supplier

#### Similar to the previous dataset there are also comma's presesnt in the Quantity column

In [ ]:
# Renaming columns
data_supplier=data_supplier.rename(columns={
    "Lead Time On Paper (in days)":"Supplier_Lead_Time",
    "Maximum Order Quantity":"Order_Quantity",
    "Current Cost/ Unit":"cost_per_unit"
})


In [ ]:
# cleaning the "quantity" column
data_supplier["Order_Quantity"] = data_supplier["Order_Quantity"].str.replace(",", "").str.strip()
# changing datatype
data_supplier["Order_Quantity"]=pd.to_numeric(data_supplier["Order_Quantity"])

In [ ]:
# replacing "Green chillies"
data_supplier=data_supplier.replace("Green Chillies","Chillies")

In [ ]:
data_supplier

##### All 5 datasets have been loaded and cleaned. This involved standardising date formats, removing duplicate header rows, stripping commas and currency symbols from numeric columns, and converting all relevant columns to their correct data types (datetime, float, int).The datasets are now ready for integration and analysis.

#### Weekly Demand Engineering

##### Rather than working with individual order transactions, we aggregate demand to a weekly level. 

In [ ]:
# making each week start on Mondays
historical_order['week_start'] = (
    historical_order['order_date'] - 
    pd.to_timedelta(historical_order['order_date'].dt.weekday, unit='d')
)

weekly_demand = historical_order.groupby(
    ['week_start', 'Item']
)['Quantity'].sum().reset_index()
weekly_demand = weekly_demand.sort_values(['Item', 'week_start'])

### Forecasting future Weekly demand per Ingredient
For this project we will be forecasting the orders for the next 4 weeks

In [ ]:
# specifying the number of weeks we will be forecasting for
forecast_weeks = 4
# An array to store our results from forecasting
forecast_results = []

for item in weekly_demand['Item'].unique():
    # Accessing the data for each ingredient
    item_data = weekly_demand[weekly_demand['Item'] == item].copy()
    item_data = item_data.set_index('week_start')['Quantity']
    # forcing the data into a weekly frequency and filling any missing values
    item_data = item_data.asfreq('W-MON', method='pad')  
    
    # model development
    model = ExponentialSmoothing(
        item_data,
        trend='add',
        seasonal='add',
        seasonal_periods=4  # monthly seasonality within quarters
    ).fit()
    # applying the forecasting model
    forecast = model.forecast(forecast_weeks)
    # looping through the forecasted results
    for week, qty in zip(forecast.index, forecast.values):
        forecast_results.append({
            'Item': item,
            'week_start': week,
            'forecasted_quantity': round(max(qty, 0))  # no negative orders
        })
# converting the results to a dataframe
forecast_df = pd.DataFrame(forecast_results)

### Calculate Recommended Order Quantity

The formula is: Safety Stock = Z × std_of_demand × √(lead_time)
Recommended Order = forecasted_quantity + safety_stock
Where Z = 1.65 covers ~95% of demand scenarios (standard for inventory management).

In [ ]:
# Computing standard deviation of historical demand per item
demand_std = weekly_demand.groupby('Item')['Quantity'].std().reset_index()
demand_std.columns = ['Item', 'demand_std']

# Get average lead time per item from historical_order
avg_lead_time = historical_order.groupby('Item')['Lead Time'].apply(
    lambda x: pd.to_numeric(x, errors='coerce').mean()
).reset_index()
avg_lead_time.columns = ['Item', 'avg_lead_time']

# Merge both datasets into forecast dataset
order_plan = forecast_df.merge(demand_std, on='Item').merge(avg_lead_time, on='Item')

# Calculate safety stock and recommended order
Z = 1.65  # 95% service level

#we will set the safety stock to 20%
# This is industry standard for food service (accounts for ~1 week of variance)

order_plan['safety_stock'] = (order_plan['forecasted_quantity'] * 0.20).round(0)
order_plan['recommended_order'] = (order_plan['forecasted_quantity'] + order_plan['safety_stock']).round(0)


### Calculating How much money can be saved by using the recommended orders

In [ ]:
# Split item_cost into holding and waste and renaming the columns
holding_cost = item_cost[item_cost['cost_type'] == 'holding'][['ingredient', 'cost_per_unit']].rename(
    columns={'ingredient': 'Item', 'cost_per_unit': 'holding_cost'})

waste_cost = item_cost[item_cost['cost_type'] == 'waste'][['ingredient', 'cost_per_unit']].rename(
    columns={'ingredient': 'Item', 'cost_per_unit': 'waste_cost'})

# Get historical average weekly order per item
historical_avg = weekly_demand.groupby('Item')['Quantity'].mean().reset_index()
historical_avg.columns = ['Item', 'historical_avg_weekly']

# Build savings dataframe from order_plan
savings_df = order_plan[['Item', 'week_start', 'recommended_order']].merge(
    historical_avg, on='Item').merge(
    holding_cost, on='Item').merge(
    waste_cost, on='Item')

# Units saved = historical average - what we recommend
savings_df['units_saved'] = (savings_df['historical_avg_weekly'] - savings_df['recommended_order']).round(0)

# 30% of excess becomes actual waste
waste_rate = 0.30
savings_df['holding_savings'] = (savings_df['units_saved'] * savings_df['holding_cost']).round(2)
savings_df['waste_savings'] = (savings_df['units_saved'] * waste_rate * savings_df['waste_cost']).round(2)
savings_df['total_savings'] = (savings_df['holding_savings'] + savings_df['waste_savings']).round(2)

# Final summary
final_summary = savings_df.groupby('Item').agg(
    avg_historical_order=('historical_avg_weekly', 'mean'),
    avg_recommended_order=('recommended_order', 'mean'),
    holding_cost=('holding_cost', 'first'),
    waste_cost=('waste_cost', 'first'),
    total_holding_savings=('holding_savings', 'sum'),
    total_waste_savings=('waste_savings', 'sum'),
    total_savings=('total_savings', 'sum')
).round(2).reset_index()

# add a column to give the action to take(reduce or increase)
final_summary['action'] = final_summary['total_savings'].apply(
    lambda x: 'Reduce Order' if x > 0 else 'Increase Order')

final_summary['4_week_impact'] = final_summary['total_savings'].apply(
    lambda x: f"+₹{x:,.2f} saved" if x > 0 else f"-₹{abs(x):,.2f} risk")

print("=" * 80)
print("           QUICK SERVICE RESTAURANT INVENTORY OPTIMIZATION REPORT")
print("=" * 80)
print(final_summary[['Item', 'avg_historical_order', 'avg_recommended_order',
                       'action', 'total_holding_savings',
                       'total_waste_savings', '4_week_impact']].to_string(index=False))
print("=" * 80)
print(f"💰 Total Savings (4 weeks):  ₹{final_summary[final_summary['total_savings']>0]['total_savings'].sum():,.2f}")
print(f"⚠️  Stockout Risk:            ₹{abs(final_summary[final_summary['total_savings']<0]['total_savings'].sum()):,.2f}")
print(f"🏆 Net Client Value:          ₹{final_summary['total_savings'].sum():,.2f}")
print("=" * 80)

### Identifying the Best Supplier for Ingredients 

In [ ]:
# For each item we will find the best supplier for the restaurant
# The best supplier is one that has the lowest cost_per_unit among suppliers who can cover the recommended order quantity

avg_recommended = final_summary[['Item', 'avg_recommended_order']].rename(
    columns={'Item': 'Product'})

supplier_recommendation = data_supplier.merge(avg_recommended, on='Product')

# Filter suppliers who can fulfill the recommended quantity
supplier_recommendation = supplier_recommendation[
    supplier_recommendation['Order_Quantity'] >= supplier_recommendation['avg_recommended_order']
]

# Pick the cheapest supplier per item
best_supplier = supplier_recommendation.loc[
    supplier_recommendation.groupby('Product')['cost_per_unit'].idxmin()
].reset_index(drop=True)

best_supplier = best_supplier.rename(columns={'Product': 'Item'})

print("=" * 70)
print("        RECOMMENDED SUPPLIER PER INGREDIENT")
print("=" * 70)
print(best_supplier[['Item', 'Supplier', 'Supplier_Lead_Time', 'Order_Quantity', 'cost_per_unit']].to_string(index=False))
print("=" * 70)

#### In the summary above we can see that there is no supplier for garlic because no supplier could meet the requirement so we
will pick the supplier with the highest Order_Quantity (best effort) among the Garlic suppliers.

In [ ]:
# we separate items that have a valid supplier vs those that don't
covered_items = best_supplier['Item'].tolist()
all_items = final_summary['Item'].tolist()
missing_items = [i for i in all_items if i not in covered_items]

print(f"food items that don't have a supplier to fufill the recommended order: {missing_items}")

# For missing items we will pick the supplier with  the highest Order_Quantity
fallback_supplier = data_supplier[data_supplier['Product'].isin(missing_items)].copy()
fallback_supplier = fallback_supplier.rename(columns={'Product': 'Item'})

fallback_best = fallback_supplier.merge(avg_recommended.rename(columns={'Product':'Item'}), on='Item')
fallback_best = fallback_best.loc[
    fallback_best.groupby('Item')['Order_Quantity'].idxmax()
].reset_index(drop=True)

# A note to split order between different suppliers
fallback_best['note'] = 'consider splitting order'

# Combine into one final supplier table
best_supplier['note'] = 'Adequate'
final_supplier = pd.concat([best_supplier, fallback_best], ignore_index=True)

print("\n" + "=" * 80)
print("        FINAL SUPPLIER RECOMMENDATION")
print("=" * 80)
print(final_supplier[['Item', 'Supplier', 'Supplier_Lead_Time', 
                        'Order_Quantity', 'cost_per_unit', 'note']].to_string(index=False))
print("=" * 80)

## Final Summary

In [ ]:
# Merge final summary with supplier recommendation
final_report = final_summary.merge(
    final_supplier[['Item', 'Supplier', 'Supplier_Lead_Time', 'Order_Quantity', 'cost_per_unit', 'note']],
    on='Item'
)

# Rename for clarity
final_report = final_report.rename(columns={
    'avg_historical_order'   : 'historical_weekly_avg',
    'avg_recommended_order'  : 'recommended_weekly_order',
    'total_holding_savings'  : 'holding_savings_4wk',
    'total_waste_savings'    : 'waste_savings_4wk',
    'total_savings'          : 'net_savings_4wk',
    'Supplier'               : 'recommended_supplier',
    'Supplier_Lead_Time'     : 'lead_time_days',
    'Order_Quantity'         : 'supplier_max_quantity',
    'cost_per_unit'          : 'supplier_cost_per_unit',
    'note'                   : 'fulfillment_status'
})

# Add metadata
final_report['report_generated'] = pd.Timestamp.today().date()
final_report['forecast_horizon'] = '4 weeks'

# Print final view
print("=" * 90)
print("                    QUICK SERVICE RESTAURANT — FULL INVENTORY OPTIMIZATION REPORT")
print("=" * 90)
print(final_report[[
    'Item', 'historical_weekly_avg', 'recommended_weekly_order',
    'action', '4_week_impact',
    'recommended_supplier', 'lead_time_days', 'fulfillment_status'
]].to_string(index=False))
print("=" * 90)
print(f"Total Savings (4 weeks):   ₹{final_summary[final_summary['total_savings']>0]['total_savings'].sum():,.2f}")
print(f"Stockout Risk (Garlic):    ₹{abs(final_summary[final_summary['total_savings']<0]['total_savings'].sum()):,.2f}")
print(f"Net Client Value:           ₹{final_summary['total_savings'].sum():,.2f}")
print(f"Report Date:                {pd.Timestamp.today().date()}")
print("=" * 90)


## Data Visualization

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Quick Service Restaurant- Inventory Optimization Report', fontsize=16, fontweight='bold', y=1.02)

items = final_summary['Item'].tolist()
colors_action = ['#2ecc71' if x > 0 else '#e74c3c' for x in final_summary['total_savings']]

# ── Chart 1: Historical vs Recommended Order ──
x = range(len(items))
width = 0.35
axes[0].bar([i - width/2 for i in x], final_summary['avg_historical_order'], 
            width, label='Historical Avg', color='#3498db', alpha=0.8)
axes[0].bar([i + width/2 for i in x], final_summary['avg_recommended_order'], 
            width, label='Recommended', color='#2ecc71', alpha=0.8)
axes[0].set_title('Historical vs Recommended\nWeekly Order', fontweight='bold')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(items)
axes[0].set_ylabel('Units')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
axes[0].legend()

# ── Chart 2: Savings Breakdown ──
bar_hold = axes[1].bar(items, final_summary['total_holding_savings'], 
                        label='Holding Savings', color='#3498db', alpha=0.8)
bar_waste = axes[1].bar(items, final_summary['total_waste_savings'], 
                         bottom=final_summary['total_holding_savings'],
                         label='Waste Savings', color='#e67e22', alpha=0.8)
axes[1].set_title('Savings Breakdown\n(Holding + Waste)', fontweight='bold')
axes[1].set_ylabel('INR(₹)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'₹{v:,.0f}'))
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].legend()

# ── Chart 3: Net 4-Week Impact ──
bars = axes[2].bar(items, final_summary['total_savings'], color=colors_action, alpha=0.85)
axes[2].set_title('Net 4-Week Financial Impact\nper Ingredient', fontweight='bold')
axes[2].set_ylabel('INR(₹)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'₹{v:,.0f}'))
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')

# Add value labels on bars
for bar in bars:
    yval = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2, 
                  yval + (8000 if yval >= 0 else -25000),
                  f'₹{yval:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

#  Chart Footer
fig.text(0.5, -0.04, 
         f" Total Savings: ₹1,283,176  |   Stockout Risk: ₹434,693  |   Net Value: ₹848,483  |   {pd.Timestamp.today().date()}",
         ha='center', fontsize=11, fontstyle='italic', color='#2c3e50')

plt.tight_layout()
plt.savefig('Quick_service_report.png', dpi=150, bbox_inches='tight')
plt.show()


## Business Insights & Recommendations

Having built and validated the inventory optimization model, we now translate 
the numbers into actionable business decisions. The goal is not just to report 
what the data says but to tell the restaurant exactly what to do and why.

### Insight 1 — The Restaurant Has a Waste Problem, Not a Demand Problem

Three out of four ingredients (Chillies, Oil, Potatoes) are being over-ordered 
consistently. This is not a demand issue — customers are still buying. It is a 
planning issue. The restaurant has been ordering based on habit rather than data, 
leading to predictable, avoidable waste every single week.

**Business Action:** Adopt data-driven weekly order quantities immediately. 
The savings are not a projection — they are recoverable money being lost right now.

### Insight 2 — Garlic is a Hidden Time Bomb

While the other ingredients are over-ordered, Garlic tells the opposite story. 
Demand has been quietly growing but the restaurant has not adjusted its ordering 
to match. This is the most dangerous type of inventory problem because it is 
invisible until the day the kitchen runs out mid-service.

A Garlic stockout does not just mean lost sales on one item — in a quick service 
restaurant, a single missing ingredient can take multiple menu items off the menu 
entirely, causing a cascade of lost revenue and customer dissatisfaction.

**Business Action:** Increase Garlic order to 232,623 units per week immediately 
and split the order across Lassie Lasoon, Lasoon Joshi and Poondu Ram to ensure 
supply continuity.

### Insight 3 — Supplier Strategy Needs an Upgrade

Currently the restaurant appears to be ordering from multiple suppliers without 
a consistent selection framework. Our analysis shows that for 3 out of 4 
ingredients, a single preferred supplier can fulfill the full recommended order 
at the lowest unit cost. Consolidating to preferred suppliers reduces 
administrative overhead, strengthens supplier relationships, and lowers cost.

The exception is Garlic — where supply risk is high enough that diversifying 
across 3 suppliers is the right strategy.

**Business Action:** Establish preferred supplier agreements with Munni Mirchi 
(Chillies), AirTeli (Oil) and Aluwahlia & Sons (Potatoes). For Garlic, maintain 
relationships with all 3 suppliers as a risk management strategy.

### Insight 4 — The Real ROI of Data-Driven Inventory

| Metric | Value |
|---|---|
| Waste reduction savings (4 weeks) | ₹1,283,176 |
| Stockout risk prevented | ₹434,693 |
| Net value delivered | ₹848,483 |
| Projected  annual  value | ~₹11,000,000+ |

These numbers are calculated conservatively — using only 4 ingredients over 
4 weeks. A full implementation across the entire menu and a 52-week horizon 
would multiply this impact significantly.

For context: the cost of building this system is a one-time investment. 
The savings it generates are recurring — every single week.

### Insight 5 — What to Do Next

| Priority | Action | Expected Impact |
|---|---|---|
| 🔴 Immediate | Increase Garlic order to 232k units/week | Prevent imminent stockout risk |
| 🔴 Immediate | Reduce Chillies, Oil & Potatoes to recommended quantities | Begin recovering ₹1.28M in waste savings |
| 🟡 Short-term | Extend forecast to 52 weeks | Build annual savings case (~₹11M) |
| 🟡 Short-term | Integrate sales data for demand validation | Improve forecast accuracy |
| 🟢 Long-term | Connect to live POS system | Real-time reorder alerts & automation |
| 🟢 Long-term | Expand model to full ingredient list | Maximize waste reduction across entire menu |

The foundation has been built. The data infrastructure, forecasting engine and 
financial model are all in place. Scaling this to the full operation is the 
logical next step.

---

## 👤 Author

**Tanimowo Possible**

🔗 [LinkedIn](https://www.linkedin.com/in/tanimowo-possible/)  
💻 [GitHub](https://github.com/positanny13/)

---